# 05 - Final Hold-out Evaluation

Stage 5 objective: freeze the selected Stage 4 candidate, train once on train+validation, and run one final unbiased hold-out test evaluation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def _find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    projects_dir = Path.home() / 'PycharmProjects'
    if projects_dir.exists():
        candidates.extend([path for path in projects_dir.iterdir() if path.is_dir()])

    for candidate in candidates:
        if (candidate / 'src' / 'config.py').is_file() and (candidate / 'data' / 'processed' / 'california_housing_clean.csv').is_file():
            return candidate

    raise RuntimeError('Could not locate project root with src/config.py and processed dataset')

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    FINAL_ERROR_ANALYSIS_FILE,
    FINAL_FEATURE_IMPORTANCE_FILE,
    FINAL_MODEL_LABEL,
    FINAL_MODEL_SUMMARY_FILE,
    FINAL_TEST_METRICS_CSV_FILE,
    FINAL_TEST_METRICS_JSON_FILE,
    PROCESSED_DATA_FILE,
    SPLIT_METADATA_FILE,
    TARGET_COLUMN,
    TUNED_RESULTS_FILE,
)
from src.data import load_processed_dataset, save_json_report, save_report_dataframe
from src.evaluate import absolute_error_by_target_quantile, build_residual_frame, residual_summary
from src.finalize import (
    build_frozen_final_pipeline,
    extract_feature_importance,
    get_frozen_final_model_config,
    load_stage4_validation_reference,
    make_final_dev_set,
    save_final_model_artifact,
)
from src.split import create_train_val_test_split, load_split_metadata

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.style.use('ggplot')

## 1. Load Processed Data and Frozen Split Metadata

In [ ]:
df = load_processed_dataset(PROCESSED_DATA_FILE)
split_metadata = load_split_metadata(SPLIT_METADATA_FILE)

print(f'Processed dataset: {PROCESSED_DATA_FILE.resolve()}')
print(f'Split metadata: {SPLIT_METADATA_FILE.resolve()}')
print(f'Dataset shape: {df.shape}')
pd.Series(split_metadata, name='value')

**Protocol note:** the same deterministic split logic from Stage 3/4 is reused; no new split strategy is introduced.

## 2. Freeze Final Model Choice (No New Selection)

In [ ]:
frozen_config = get_frozen_final_model_config()
stage4_val_reference = load_stage4_validation_reference(path=TUNED_RESULTS_FILE, model_label=FINAL_MODEL_LABEL)

print('Frozen model label:', frozen_config.model_label)
print('Frozen model family:', frozen_config.model_family)
print('Frozen model params:', frozen_config.model_params)
print('Frozen add_stage4_features:', frozen_config.add_stage4_features)

pd.Series(stage4_val_reference, name='Stage4_validation_metric')

**Frozen decision:** final model remains `GradientBoostingRegressor_tuned` with Stage 4-selected hyperparameters and `add_stage4_features=False`.

## 3. Recreate Split and Build Final Development Set (Train + Validation)

In [ ]:
split = create_train_val_test_split(
    df,
    target_column=TARGET_COLUMN,
    test_size=float(split_metadata['test_size']),
    val_size=float(split_metadata['validation_size']),
    random_state=int(split_metadata['random_state']),
)

X_dev, y_dev = make_final_dev_set(
    X_train=split.X_train,
    X_val=split.X_val,
    y_train=split.y_train,
    y_val=split.y_val,
)

print('Development set (train + val):', X_dev.shape, y_dev.shape)
print('Hold-out test set          :', split.X_test.shape, split.y_test.shape)

**Unbiased final-training design:** model is fitted once on development data (`train + validation`) and evaluated once on untouched test data.

## 4. Fit Frozen Final Pipeline and Run One Hold-out Test Evaluation

In [ ]:
final_pipeline = build_frozen_final_pipeline()
final_pipeline.fit(X_dev, y_dev)

# Final hold-out prediction (single frozen model; no test-set comparisons)
y_test_pred = final_pipeline.predict(split.X_test)

rmse_test = float(mean_squared_error(split.y_test, y_test_pred) ** 0.5)
mae_test = float(mean_absolute_error(split.y_test, y_test_pred))
r2_test = float(r2_score(split.y_test, y_test_pred))

test_metrics = {
    'RMSE_test': rmse_test,
    'MAE_test': mae_test,
    'R2_test': r2_test,
}
pd.Series(test_metrics, name='Final_test_metric')

In [ ]:
comparison_df = pd.DataFrame(
    [
        {'Dataset': 'Validation (Stage 4)', **stage4_val_reference},
        {'Dataset': 'Test (Stage 5)', **{
            'RMSE_val': test_metrics['RMSE_test'],
            'MAE_val': test_metrics['MAE_test'],
            'R2_val': test_metrics['R2_test'],
        }},
    ]
)
comparison_df = comparison_df.rename(columns={'RMSE_val': 'RMSE', 'MAE_val': 'MAE', 'R2_val': 'R2'})
comparison_df['RMSE_gap_test_minus_val'] = comparison_df['RMSE'].diff()
comparison_df['MAE_gap_test_minus_val'] = comparison_df['MAE'].diff()
comparison_df['R2_gap_test_minus_val'] = comparison_df['R2'].diff()
comparison_df

**Generalization check:** compare Stage 4 validation metrics against final test metrics to verify stability without reopening model selection.

## 5. Save Final Model and Final Metric Reports

In [ ]:
model_path = save_final_model_artifact(final_pipeline)
print(f'Final model artifact saved to: {model_path.resolve()}')

metrics_payload = {
    'model_label': frozen_config.model_label,
    'model_family': frozen_config.model_family,
    'frozen_params': frozen_config.model_params,
    'add_stage4_features': frozen_config.add_stage4_features,
    'validation_reference': stage4_val_reference,
    'final_test_metrics': test_metrics,
    'gaps_test_minus_validation': {
        'RMSE_gap': float(test_metrics['RMSE_test'] - stage4_val_reference['RMSE_val']),
        'MAE_gap': float(test_metrics['MAE_test'] - stage4_val_reference['MAE_val']),
        'R2_gap': float(test_metrics['R2_test'] - stage4_val_reference['R2_val']),
    },
}

save_json_report(metrics_payload, FINAL_TEST_METRICS_JSON_FILE)
metrics_table = pd.DataFrame([
    {
        'model_label': frozen_config.model_label,
        'RMSE_test': test_metrics['RMSE_test'],
        'MAE_test': test_metrics['MAE_test'],
        'R2_test': test_metrics['R2_test'],
        'RMSE_val_stage4': stage4_val_reference['RMSE_val'],
        'MAE_val_stage4': stage4_val_reference['MAE_val'],
        'R2_val_stage4': stage4_val_reference['R2_val'],
    }
])
save_report_dataframe(metrics_table, FINAL_TEST_METRICS_CSV_FILE)
metrics_table

## 6. Final Error Analysis on Hold-out Test

In [ ]:
residual_test = build_residual_frame(split.y_test, y_test_pred)
residual_stats = residual_summary(residual_test)
error_by_bin = absolute_error_by_target_quantile(residual_test, bins=5)

save_report_dataframe(error_by_bin, FINAL_ERROR_ANALYSIS_FILE)

print(f'Final error analysis saved to: {FINAL_ERROR_ANALYSIS_FILE.resolve()}')
display(residual_stats.to_frame(name='value'))
display(error_by_bin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(residual_test['actual'], residual_test['predicted'], alpha=0.25, s=12)
min_val = min(residual_test['actual'].min(), residual_test['predicted'].min())
max_val = max(residual_test['actual'].max(), residual_test['predicted'].max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=1)
axes[0].set_title('Predicted vs Actual (Final Test)')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

axes[1].scatter(residual_test['predicted'], residual_test['residual'], alpha=0.25, s=12, color='darkorange')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Residuals vs Predicted (Final Test)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

**Error-analysis interpretation:** check whether absolute error increases in high target bins and confirm if expensive homes remain the hardest segment.

## 7. Lightweight Interpretation: Feature Importances

In [ ]:
feature_importance_df = extract_feature_importance(final_pipeline)
save_report_dataframe(feature_importance_df, FINAL_FEATURE_IMPORTANCE_FILE)

print(f'Final feature importances saved to: {FINAL_FEATURE_IMPORTANCE_FILE.resolve()}')
feature_importance_df.head(15)

In [ ]:
top_n = 12
plot_frame = feature_importance_df.head(top_n).iloc[::-1]

plt.figure(figsize=(8, 5))
plt.barh(plot_frame['feature'], plot_frame['importance'])
plt.title(f'Top {top_n} Feature Importances (Final Model)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

**Interpretation goal:** identify whether model relies mainly on income and location signals and how ratio features compare to core original predictors.

## 8. Write Final Model Summary Report

In [ ]:
top_bin_row = error_by_bin.iloc[-1]
summary_text = f"""# Final Model Summary

## Frozen Final Choice
- Model label: `{frozen_config.model_label}`
- Model family: `{frozen_config.model_family}`
- Hyperparameters: `{frozen_config.model_params}`
- `add_stage4_features`: `{frozen_config.add_stage4_features}`

## Final Hold-out Test Metrics
- RMSE_test: `{test_metrics['RMSE_test']:.6f}`
- MAE_test: `{test_metrics['MAE_test']:.6f}`
- R2_test: `{test_metrics['R2_test']:.6f}`

## Validation vs Test Stability
- Stage 4 validation RMSE: `{stage4_val_reference['RMSE_val']:.6f}`
- Final test RMSE: `{test_metrics['RMSE_test']:.6f}`
- RMSE gap (test - validation): `{test_metrics['RMSE_test'] - stage4_val_reference['RMSE_val']:.6f}`

## Error Behavior
- Residual mean: `{float(residual_stats['residual_mean']):.6f}`
- Highest target-bin mean absolute error: `{float(top_bin_row['abs_error_mean']):.6f}` for bin `{top_bin_row['target_bin']}`
- Expensive homes remain the hardest segment if the top-bin error is largest.

## Interpretation
- Top features are listed in `reports/final_feature_importance.csv`.
- This report supports a reproducible and interview-ready final model handoff.
"""

FINAL_MODEL_SUMMARY_FILE.write_text(summary_text, encoding='utf-8')
print(f'Final model summary saved to: {FINAL_MODEL_SUMMARY_FILE.resolve()}')

## Stage 5 Final Conclusions

- Final model is frozen and evaluated once on hold-out test data.
- Final reporting package is saved in `reports/` and model artifact in `models/`.
- Project now has a full, reproducible end-to-end ML workflow suitable for portfolio and interview discussion.